# Qwen3-8B ecological-dilemma fine-tuning

This notebook fine-tunes `Qwen/Qwen3-8B` on one of three 98-case ecological-versus-human dilemma arms selected in the configuration cell: `prompt_only`, `ecological_option`, or `human_option`. The prompt-only arm retains the original user-only causal objective. The two answer arms pair each dilemma with the exact corresponding option field as a one-turn assistant response, mask the user turn, and apply loss only to that option text and its terminating token. No arm contains a rationale. The current default is `ecological_option`.

Training completes on local `/content` storage, saves resumable epoch checkpoints and the final LoRA adapter, then copies the entire hash-verified run to Google Drive, flushes and unmounts Drive, freshly remounts it, and verifies every required hash. The saved adapter is then evaluated against the unmodified base model on the eight primary `extreme_v2` prompts and all six controls. Both verified evaluation bundles are published to GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
# Colab's optional TorchAO build can conflict with PEFT. This workflow uses
# ordinary BF16 LoRA and does not use TorchAO quantization.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
REPOSITORY_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This workflow requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

## Configuration

The defaults retain the proven H4rmony Qwen LoRA setup where it transfers: BF16, rank 16, alpha 32, dropout 0.05, all linear layers, three epochs, micro-batch size 1, gradient accumulation 16, and seed 42. Change only `TRAINING_ARM` to select the intervention. Dataset paths and isolated local/Drive output roots follow automatically.

In [ ]:
from google.colab import userdata
from scripts.ecological_prompt_sft import (
    DEFAULT_DATASET_PATHS,
    TRAINING_ARMS,
    DilemmaSFTConfig,
)

TRAINING_ARM = "ecological_option"  # prompt_only | ecological_option | human_option
assert TRAINING_ARM in TRAINING_ARMS
OUTPUT_SLUGS = {
    "prompt_only": "ecological_dilemma_prompt_qwen3_8b",
    "ecological_option": "ecological_dilemma_ecological_option_qwen3_8b",
    "human_option": "ecological_dilemma_human_option_qwen3_8b",
}
OUTPUT_SLUG = OUTPUT_SLUGS[TRAINING_ARM]
LOCAL_OUTPUT_ROOT = Path("/content/value-misalignment-runs") / OUTPUT_SLUG
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/value-misalignment") / OUTPUT_SLUG
FORCE_RETRAIN = False
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"

CONFIG = DilemmaSFTConfig(
    output_root=LOCAL_OUTPUT_ROOT,
    training_arm=TRAINING_ARM,
    dataset_path=DEFAULT_DATASET_PATHS[TRAINING_ARM],
    base_model="Qwen/Qwen3-8B",
    model_revision="b968826d9c46dd6066d109eabc6255188de91218",
    max_length=1024,
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    eval_batch_size=4,
    seed=42,
    cost_counts=(0, 1, 10, 100, 1_000, 10_000, 100_000, 1_000_000),
)

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add a Colab secret named GITHUB_TOKEN and grant this notebook access before training."
        ) from exc
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN is empty; publication is enabled and required by this workflow."
        )
print("Training arm:", CONFIG.training_arm)
print("Dataset:", CONFIG.dataset_path)
print("N values:", CONFIG.cost_counts)
CONFIG

In [ ]:
from IPython.display import Markdown, display
from scripts.ecological_prompt_sft import load_training_examples

training_examples, training_manifest = load_training_examples(
    CONFIG.dataset_path,
    training_arm=CONFIG.training_arm,
)
assert len(training_examples) == 98
assert training_manifest["training_arm"] == CONFIG.training_arm
print("Example count:", len(training_examples))
print("records.jsonl SHA-256:", training_manifest["records_sha256"])
if CONFIG.training_arm == "prompt_only":
    assert training_manifest["contains_normative_labels"] is False
    assert training_manifest["contains_assistant_responses"] is False
    print("Prompt-only arm: no assistant responses. First five examples:")
else:
    assert training_manifest["contains_normative_labels"] is True
    assert training_manifest["contains_assistant_responses"] is True
    assert training_manifest["contains_rationales"] is False
    print("Assistant target field:", training_manifest["assistant_target_field"])
    print("Response-only loss; no rationales. First five examples:")
for example in training_examples[:5]:
    preview = f"### `{example['id']}` — {example['title']}\n\n**User**\n\n{example['dilemma']}"
    if "assistant_answer" in example:
        preview += f"\n\n**Assistant**\n\n{example['assistant_answer']}"
    display(Markdown(preview))

In [ ]:
from scripts.ecological_prompt_sft import (
    find_compatible_complete_run,
    persist_run_to_colab_drive,
    run_dilemma_sft,
)

artifacts = None
if not FORCE_RETRAIN:
    print(f"Checking Drive for a compatible, hash-verified {CONFIG.training_arm} run...")
    artifacts = find_compatible_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)

if artifacts is not None:
    print("TRAINING SKIPPED — reusing verified Drive run:", artifacts.run_dir)
else:
    print(f"Starting {CONFIG.training_arm} Qwen fine-tuning on all 98 dilemmas.")
    local_artifacts = run_dilemma_sft(CONFIG)
    artifacts = persist_run_to_colab_drive(local_artifacts, DRIVE_OUTPUT_ROOT)
    print("Completed and freshly verified Drive run:", artifacts.run_dir)

In [ ]:
import json
import pandas as pd

complete = json.loads(artifacts.complete_marker_path.read_text())
train_metrics = json.loads(artifacts.train_metrics_path.read_text())
dataset_manifest = json.loads(artifacts.dataset_manifest_path.read_text())
assert complete["status"] == "complete"
assert dataset_manifest["example_count"] == 98
assert dataset_manifest["training_arm"] == CONFIG.training_arm
assert dataset_manifest["contains_normative_labels"] == (CONFIG.training_arm != "prompt_only")
assert dataset_manifest["contains_assistant_responses"] == (CONFIG.training_arm != "prompt_only")
display(pd.DataFrame([train_metrics]))
display(pd.DataFrame([dataset_manifest["tokenization"]["sequence_tokens"]]))
display(pd.DataFrame([dataset_manifest["tokenization"]["supervised_tokens"]]))
print("Training arm:", train_metrics["training_arm"])
print("Training objective:", train_metrics["training_objective"])
print("Final adapter:", artifacts.final_adapter_dir)
print("Resumable checkpoints and metadata:", artifacts.run_dir)

In [ ]:
from scripts.ecological_prompt_sft import (
    EXTREME_V2_TEMPLATES,
    build_extreme_v2_cases,
)

primary_preview_cases = build_extreme_v2_cases(CONFIG.cost_counts)
assert len(primary_preview_cases) == len(EXTREME_V2_TEMPLATES) * len(CONFIG.cost_counts)
print(f"Reviewing all {len(primary_preview_cases)} primary cases before inference.")
for case in primary_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
from scripts.ecological_prompt_sft import (
    EXTREME_V2_CONTROL_TEMPLATES,
    build_extreme_v2_control_cases,
)

control_preview_cases = build_extreme_v2_control_cases(CONFIG.cost_counts)
assert len(control_preview_cases) == 4 * len(CONFIG.cost_counts) + 2
print(f"Reviewing all {len(control_preview_cases)} control cases before inference.")
for case in control_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
from scripts.ecological_prompt_sft import run_extreme_v2_workflow

primary_workflow = run_extreme_v2_workflow(
    artifacts,
    cost_counts=CONFIG.cost_counts,
    batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
if primary_workflow.evaluation_reused:
    print("PRIMARY EVALUATION SKIPPED — reusing verified Drive results.")
else:
    print("PRIMARY EVALUATION COMPLETE — verified after a fresh Drive remount.")
primary_workflow.validation

In [ ]:
from IPython.display import Image

primary_eval = primary_workflow.evaluation_artifacts
primary_scores = pd.read_csv(primary_eval.raw_scores_path)
assert len(primary_scores) == 2 * len(primary_preview_cases)
assert set(primary_scores["model_role"]) == {"base", "aligned"}
display(primary_scores.pivot(
    index=["template", "cost_count"],
    columns="model_role",
    values=["p_implement", "semantic_logit_implement"],
).sort_index())
display(pd.read_csv(primary_eval.thresholds_path))
display(Image(filename=str(primary_eval.plot_path)))
print("Verified primary Drive bundle:", primary_eval.output_dir)

In [ ]:
from scripts.ecological_prompt_sft import run_extreme_v2_control_workflow

control_workflow = run_extreme_v2_control_workflow(
    artifacts,
    cost_counts=CONFIG.cost_counts,
    batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
if control_workflow.evaluation_reused:
    print("CONTROL EVALUATION SKIPPED — reusing verified Drive results.")
else:
    print("CONTROL EVALUATION COMPLETE — verified after a fresh Drive remount.")
control_workflow.validation

In [ ]:
control_eval = control_workflow.evaluation_artifacts
control_scores = pd.read_csv(control_eval.raw_scores_path)
assert len(control_scores) == 2 * len(control_preview_cases)
assert set(control_scores["model_role"]) == {"base", "aligned"}
display(control_scores.pivot(
    index=["template", "cost_count"],
    columns="model_role",
    values=["p_implement", "semantic_logit_implement"],
).sort_index())
display(pd.read_csv(control_eval.thresholds_path))
display(Image(filename=str(control_eval.plot_path)))
print("Verified control Drive bundle:", control_eval.output_dir)

In [ ]:
from scripts.ecological_prompt_sft import publish_results_to_github

if PUBLISH_TO_GITHUB:
    primary_publication = publish_results_to_github(
        primary_eval,
        source_run_name=artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY,
        branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN,
        repo_root=REPO_DIR,
    )
    print("Primary GitHub publication verified:", primary_publication.html_url)
    control_publication = publish_results_to_github(
        control_eval,
        source_run_name=artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY,
        branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN,
        repo_root=REPO_DIR,
    )
    print("Control GitHub publication verified:", control_publication.html_url)
else:
    print("GitHub publication disabled; both verified Drive bundles remain intact.")